# Tutorial 2: Singularity detectors

Hyperbolic growth `dx/dt = k x^(1+δ)` reaches infinity at a finite date `t_c`:

`x(t) = x0 (1 − t/t_c)^(−1/δ)`

The tell is in the growth rate: `ln g = ln k + δ ln x`. Exponential growth has
`δ = 0`. This notebook estimates `δ` and `t_c` on two synthetic series, one
exponential and one hyperbolic, and puts a block-bootstrap interval on `δ`.

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:  # plots are optional
    plt = None

rng = np.random.default_rng(13)

In [ ]:
t = np.arange(0, 20, 1 / 12)                    # years
t_c_true, delta_true = 30.0, 0.5
noise = rng.normal(0, 0.02, t.size)
ln_hyper = -(1 / delta_true) * np.log(1 - t / t_c_true) + noise
ln_expo = 0.15 * t + noise

## Estimate `g` from history only

`g` comes from a smoothed derivative of the *observed* data, never from a
forecast segment (D1). Here a local quadratic over a 2-year window.

In [ ]:
def local_slope(t, y, half_width=1.0):
    g = np.full_like(y, np.nan)
    for i, ti in enumerate(t):
        w = np.abs(t - ti) <= half_width
        if w.sum() >= 5:
            g[i] = np.polyfit(t[w] - ti, y[w], 2)[1]
    return g


def fit_delta(t, ln_x):
    g = local_slope(t, ln_x)
    ok = np.isfinite(g) & (g > 0)
    delta, ln_k = np.polyfit(ln_x[ok], np.log(g[ok]), 1)
    g_now = g[ok][-1]
    t_rem = 1 / (delta * g_now) if delta > 0 else np.inf
    return delta, t_rem


for name, series in [("exponential", ln_expo), ("hyperbolic", ln_hyper)]:
    d, rem = fit_delta(t, series)
    print(f"{name:12s} delta = {d:+.3f}   time remaining = {rem:.1f} yr")
print(f"truth: delta = {delta_true}, time remaining = {t_c_true - t[-1]:.1f} yr")

`δ` comes out close to the truth, but the time remaining is too long. The
smoother underestimates `g` at the last point (an edge effect), and
`1/(δ g)` amplifies that. This is why the library also fits
`x = x0 (1 − t/t_c)^(−1/δ)` directly and reports both (K-5.03).

## Honest uncertainty: moving block bootstrap

`ln g` and `ln x` come from the same autocorrelated series, so the regression's
own standard error is far too small (D4). Resample blocks of residuals around
a smooth fit and refit the whole procedure each time.

In [ ]:
def block_bootstrap_delta(t, ln_x, n_boot=200, block=12):
    coef = np.polyfit(t, ln_x, 5)
    smooth = np.polyval(coef, t)
    resid = ln_x - smooth
    n = t.size
    out = []
    for _ in range(n_boot):
        starts = rng.integers(0, n - block, size=n // block + 1)
        idx = np.concatenate([np.arange(s, s + block) for s in starts])[:n]
        out.append(fit_delta(t, smooth + resid[idx])[0])
    return np.array(out)


boot = block_bootstrap_delta(t, ln_hyper)
lo, hi = np.quantile(boot, [0.05, 0.95])
print(f"hyperbolic delta 90% interval: [{lo:.2f}, {hi:.2f}]  P(delta > 0) = {(boot > 0).mean():.2f}")
if plt is not None:
    plt.hist(boot, bins=30)
    plt.axvline(0, color="k")
    plt.title("bootstrap distribution of delta")
    plt.show()

## LPPLS in one line

Sornette's log-periodic power law adds accelerating oscillations:

`ln x = A + B (t_c − t)^m + C (t_c − t)^m cos(ω ln(t_c − t) − φ)`

In the library the `(t_c, m, ω)` grid is searched on the GPU and the linear
parameters are solved in closed form for every grid cell (K-5.06 to K-5.09).